# Zajęcia 4: GraphRAG i grafy wiedzy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/04-graphrag/notebook.ipynb)

Zakres:
1. Graf typu *property graph* w Neo4j: węzły, relacje, właściwości.
2. Składnia języka Cypher na przykładach.
3. Ekstrakcja grafu wiedzy z tekstu przez LLM: schemat Pydantic i structured outputs.
4. Spójność encji („K8s” i „Kubernetes” to jeden węzeł).
5. Graf TalentMatch: 14 CV i 3 zapytania ofertowe (RFP), dopasowanie kandydatów.
6. Agentic GraphRAG: agent w LangGraph z narzędziem do zapytań Cypher.
7. Porównanie trzech podejść: RAG, structured RAG (filtr po metadanych + wyszukiwanie wektorowe) i GraphRAG.

Komórki z tagiem `web:*` są pokazywane na prezentacji.

## 0. Instalacja i konfiguracja

In [1]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai neo4j pydantic python-dotenv chromadb langchain langchain-openai langgraph
# Colab: dane TalentMatch są w repozytorium kursu
# !git clone -q --depth 1 https://github.com/adrianstando/2026Z-PJATK-TEG.git && %cd 2026Z-PJATK-TEG/04-graphrag

Dostawcę modelu wybiera się zmienną `LLM_PROVIDER` (plik `.env` albo w Colabie: ikona klucza → *Secrets*), tak jak na zajęciach 1. Instrukcja: [00-dostep-do-llm](../00-dostep-do-llm/README.md).

| `LLM_PROVIDER` | Wymagania |
|---|---|
| `azure` | `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, deploymenty `gpt-4o-mini` i `text-embedding-3-small` |
| `openai` | `OPENAI_API_KEY` |
| `deepseek` | `DEEPSEEK_API_KEY`; bez embeddingów |
| `ollama` | [Ollama](https://ollama.com) lokalnie, modele `gemma4:e4b`, `embeddinggemma` |
| `proxy` | uruchomione [proxy dla subskrypcji](../tools/teg-proxy/README.md); bez embeddingów |

Ekstrakcja grafu korzysta ze *structured outputs* (odpowiedź zgodna ze schematem JSON), a agent z wywołań narzędzi. Najpewniej działa to w OpenAI i Azure; małe modele lokalne częściej gubią pola i mylą argumenty narzędzi.

In [2]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "EMBED_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "DEEPSEEK_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

# chat: ekstrakcja (structured outputs), agent: pętla agenta i odpowiedzi RAG, embed: embeddingi.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "agent": "gpt-5.4-nano", "embed": "text-embedding-3-small"},
    "azure":    {"chat": "gpt-4o-mini", "agent": "gpt-4o-mini", "embed": "text-embedding-3-small"},
    "deepseek": {"chat": "deepseek-chat", "agent": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "agent": "gemma4:e4b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "agent": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, AGENT = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["agent"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, agent: {AGENT}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

czat: openai (gpt-5-nano, agent: gpt-5.4-nano) | embeddingi: openai (text-embedding-3-small)


### Neo4j

Graf trzymany jest w bazie Neo4j. Są dwa sposoby, żeby ją mieć:

1. **Docker Compose**, lokalnie albo w chmurze (np. GitHub Codespaces): `docker compose up -d` w folderze `04-graphrag` (plik [docker-compose.yml](docker-compose.yml)). Podgląd grafu w przeglądarce: http://localhost:7474, logowanie `neo4j` / `password123`.
2. **Darmowa instancja Neo4j AuraDB Free**: https://console.neo4j.io → *Create instance* → *Free*. Przy tworzeniu pobiera się plik `.txt` z danymi logowania: `NEO4J_URI` (`neo4j+s://…`), `NEO4J_USERNAME` i `NEO4J_PASSWORD`. Te trzy wartości trzeba wpisać do `.env` albo do *Secrets* w Colabie.

Bez tych zmiennych notebook łączy się z lokalnym Dockerem.

In [3]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD", "password123")),
    notifications_min_severity="OFF",  # bez ostrzeżeń serwera w wynikach komórek
)
driver.verify_connectivity()

def cypher(query, **params):
    """Wykonuje zapytanie Cypher i zwraca listę słowników."""
    records, _, _ = driver.execute_query(query, params)
    return [r.data() for r in records]

def run_read(query, **params):
    """To samo w transakcji tylko do odczytu: zapytanie zapisujące kończy się błędem."""
    with driver.session() as session:
        return session.execute_read(lambda tx: [r.data() for r in tx.run(query, params)])

print(cypher("CALL dbms.components() YIELD name, versions RETURN name, versions[0] AS version"))

[{'name': 'Neo4j Kernel', 'version': '2026.09.0'}, {'name': 'Cypher', 'version': '5'}]


## 1. Graf: węzły, relacje, właściwości

Neo4j przechowuje **property graph**:
- **węzeł** ma etykietę (`:Person`, `:Skill`) i właściwości (`name`, `city`),
- **relacja** ma typ (`:HAS_SKILL`), kierunek i też może mieć właściwości (`years`): lata doświadczenia są cechą połączenia osoby z umiejętnością, a nie samej osoby.

Te same fakty w bazie relacyjnej to tabele `people`, `skills`, `companies` i tabele łączące `has_skill(person_id, skill_id, years)`, `worked_at(person_id, company_id, since)`. W grafie tabela łącząca jest po prostu krawędzią.

Konwencje nazw w Cypherze: etykiety węzłów w `PascalCase`, typy relacji w `UPPER_SNAKE_CASE`, właściwości w `camelCase` albo `snake_case`.

Mały graf zbudowany ręcznie: cztery osoby, dwie umiejętności, dwie firmy. `MATCH (n) DETACH DELETE n` na początku czyści bazę (opis składni w następnej sekcji).

In [4]:
cypher("MATCH (n) DETACH DELETE n")  # pusta baza

cypher("""
CREATE (anna:Person {name: 'Anna', city: 'Warszawa'}), (piotr:Person {name: 'Piotr', city: 'Kraków'}),
       (ewa:Person {name: 'Ewa', city: 'Gdańsk'}), (marek:Person {name: 'Marek', city: 'Kraków'}),
       (py:Skill {name: 'Python'}), (k8s:Skill {name: 'Kubernetes'}),
       (nb:Company {name: 'Northbyte'}), (cf:Company {name: 'Cloudforge'}),
       (anna)-[:HAS_SKILL {years: 8}]->(py),
       (anna)-[:HAS_SKILL {years: 3}]->(k8s),
       (piotr)-[:HAS_SKILL {years: 5}]->(k8s),
       (ewa)-[:HAS_SKILL {years: 2}]->(py),
       (marek)-[:HAS_SKILL {years: 4}]->(py),
       (anna)-[:WORKED_AT {since: 2021}]->(nb),
       (ewa)-[:WORKED_AT {since: 2023}]->(nb),
       (piotr)-[:WORKED_AT {since: 2019}]->(cf),
       (marek)-[:WORKED_AT {since: 2020}]->(cf),
       (anna)-[:KNOWS]->(piotr)
""")
print(cypher("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY label"))
print(cypher("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY type"))

[{'label': 'Company', 'n': 2}, {'label': 'Person', 'n': 4}, {'label': 'Skill', 'n': 2}]
[{'type': 'HAS_SKILL', 'n': 5}, {'type': 'KNOWS', 'n': 1}, {'type': 'WORKED_AT', 'n': 4}]


## 2. Składnia Cypher

Cypher (język zapytań Neo4j) opisuje wzorzec do znalezienia w grafie: węzły w nawiasach okrągłych, relacje w kwadratowych, kierunek strzałką.

| Fragment | Znaczenie | Odpowiednik w SQL |
|---|---|---|
| `(p:Person {name: 'Anna'})` | węzeł: zmienna `p`, etykieta, właściwości do dopasowania | wiersz tabeli `people` z warunkiem |
| `-[r:HAS_SKILL]->` | relacja ze zmienną `r` i typem; `-[r]-` dowolny kierunek, `-[]->` dowolny typ | `JOIN` przez tabelę łączącą |
| `-[*1..3]-` | ścieżka od 1 do 3 relacji | rekurencyjne `WITH RECURSIVE` |
| `MATCH` | znajdź wszystkie dopasowania wzorca | `SELECT … FROM … JOIN` |
| `OPTIONAL MATCH` | jak `MATCH`, ale brak dopasowania daje `null` | `LEFT JOIN` |
| `WHERE` | filtr na właściwościach węzłów i relacji | `WHERE` |
| `RETURN … AS …` | co zwrócić i pod jaką nazwą | lista kolumn w `SELECT` |
| `count()`, `avg()`, `sum()`, `collect()` | agregacje; `collect` zbiera wartości w listę; grupuje się po pozostałych kolumnach z `RETURN` | `COUNT`, `AVG`, `GROUP BY`, `array_agg` |
| `ORDER BY`, `LIMIT` | sortowanie i limit | to samo |
| `WITH` | przekazuje wyniki do dalszej części zapytania (krok pośredni, np. filtr po agregacji) | podzapytanie, `HAVING` |
| `CREATE` | zawsze tworzy nowy węzeł albo relację | `INSERT` |
| `MERGE` | znajdź albo utwórz (po podanych właściwościach) | `INSERT … ON CONFLICT DO NOTHING` |
| `SET` | ustawia właściwości | `UPDATE … SET` |
| `DETACH DELETE` | usuwa węzeł razem z jego relacjami (samo `DELETE` nie usunie węzła, który ma relacje) | `DELETE` z kaskadą |
| `UNWIND $lista AS x` | zamienia listę na wiersze, po jednym na element | `unnest()` |
| `$param` | parametr przekazany z Pythona (bezpieczniej niż sklejanie tekstu) | `?` / `%s` w zapytaniu |
| `coalesce(a, b)` | pierwsza wartość różna od `null` | `COALESCE` |
| `[x IN lista WHERE … \| wyrażenie]` | operacja na liście (filtr i przekształcenie) | brak prostego odpowiednika |

Przykłady na małym grafie z sekcji 1 (Neo4j 5). Ostatnie dwa zapytania zapisują dane, a ostatnie usuwa to, co dodało przedostatnie.

In [5]:
EXAMPLES = [
    ("wszystkie węzły z etykietą", "MATCH (p:Person) RETURN p.name AS name, p.city AS city ORDER BY name"),
    ("filtr WHERE", "MATCH (p:Person) WHERE p.city = 'Kraków' RETURN p.name AS name"),
    ("wzorzec z relacją", "MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill {name: 'Python'})\nRETURN p.name AS name, r.years AS years ORDER BY years DESC"),
    ("filtr na właściwości relacji", "MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)\nWHERE r.years >= 4\nRETURN p.name AS name, s.name AS skill, r.years AS years"),
    ("agregacja", "MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)\nRETURN s.name AS skill, count(p) AS people, avg(r.years) AS avg_years\nORDER BY people DESC"),
    ("collect: lista w wyniku", "MATCH (p:Person)-[:WORKED_AT]->(c:Company)\nRETURN c.name AS company, collect(p.name) AS people"),
    ("dwa kroki po relacjach", "MATCH (:Person {name: 'Anna'})-[:WORKED_AT]->(c:Company)<-[:WORKED_AT]-(p:Person)\nRETURN p.name AS colleague, c.name AS company"),
    ("dowolna relacja, dowolny kierunek", "MATCH (:Person {name: 'Piotr'})-[r]-(x)\nRETURN type(r) AS rel, x.name AS neighbour"),
    ("ścieżka zmiennej długości", "MATCH path = (:Person {name: 'Ewa'})-[*1..3]-(:Person {name: 'Piotr'})\nRETURN [n IN nodes(path) | n.name] AS path"),
    ("WITH: filtr po agregacji", "MATCH (p:Person)-[:HAS_SKILL]->(s:Skill)\nWITH p, count(s) AS skills\nWHERE skills >= 2\nRETURN p.name AS name, skills"),
    ("parametr $city", "MATCH (p:Person {city: $city}) RETURN p.name AS name"),
    ("MERGE: znajdź albo utwórz", "MERGE (p:Person {name: 'Ola'}) SET p.city = 'Poznań'\nMERGE (s:Skill {name: 'Python'})\nMERGE (p)-[:HAS_SKILL {years: 1}]->(s)\nWITH s MATCH (x:Skill {name: 'Python'})<-[:HAS_SKILL]-(p:Person)\nRETURN count(DISTINCT x) AS python_nodes, count(p) AS people"),
    ("DETACH DELETE", "MATCH (p:Person {name: 'Ola'}) DETACH DELETE p\nWITH 1 AS done MATCH (n) RETURN count(n) AS nodes"),
]

for title, query in EXAMPLES:
    print(f"=== {title}\n{query}")
    print("-->", cypher(query, city="Kraków"), "\n")

=== wszystkie węzły z etykietą
MATCH (p:Person) RETURN p.name AS name, p.city AS city ORDER BY name
--> [{'name': 'Anna', 'city': 'Warszawa'}, {'name': 'Ewa', 'city': 'Gdańsk'}, {'name': 'Marek', 'city': 'Kraków'}, {'name': 'Piotr', 'city': 'Kraków'}] 

=== filtr WHERE
MATCH (p:Person) WHERE p.city = 'Kraków' RETURN p.name AS name
--> [{'name': 'Piotr'}, {'name': 'Marek'}] 

=== wzorzec z relacją
MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill {name: 'Python'})
RETURN p.name AS name, r.years AS years ORDER BY years DESC
--> [{'name': 'Anna', 'years': 8}, {'name': 'Marek', 'years': 4}, {'name': 'Ewa', 'years': 2}] 

=== filtr na właściwości relacji
MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)
WHERE r.years >= 4
RETURN p.name AS name, s.name AS skill, r.years AS years
--> [{'name': 'Piotr', 'skill': 'Kubernetes', 'years': 5}, {'name': 'Marek', 'skill': 'Python', 'years': 4}, {'name': 'Anna', 'skill': 'Python', 'years': 8}] 

=== agregacja
MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)
RETURN s.name AS 

--> [{'name': 'Piotr'}, {'name': 'Marek'}] 

=== MERGE: znajdź albo utwórz
MERGE (p:Person {name: 'Ola'}) SET p.city = 'Poznań'
MERGE (s:Skill {name: 'Python'})
MERGE (p)-[:HAS_SKILL {years: 1}]->(s)
WITH s MATCH (x:Skill {name: 'Python'})<-[:HAS_SKILL]-(p:Person)
RETURN count(DISTINCT x) AS python_nodes, count(p) AS people
--> [{'python_nodes': 1, 'people': 4}] 

=== DETACH DELETE
MATCH (p:Person {name: 'Ola'}) DETACH DELETE p
WITH 1 AS done MATCH (n) RETURN count(n) AS nodes
--> [{'nodes': 8}] 



## 3. Dane TalentMatch

Scenariusz: firma IT ma bazę CV programistów i dostaje zapytania ofertowe (RFP) od klientów. Trzeba szybko odpowiedzieć, kogo można przydzielić do projektu.

W folderze `data/` jest 14 syntetycznych CV i 3 RFP w Markdownie (`data/generate_data.py` generuje je z listy faktów). Teksty mają różną formę, a te same technologie są zapisane różnie: „K8s” i „Kubernetes”, „Postgres” i „PostgreSQL”. Każde CV ma też krótki opis najważniejszego projektu. Plik `ground_truth.json` zawiera fakty, z których powstały teksty; posłuży do sprawdzenia ekstrakcji i odpowiedzi.

In [6]:
from pathlib import Path

DATA = Path("data")
cvs = {p.stem: p.read_text(encoding="utf-8") for p in sorted((DATA / "cv").glob("*.md"))}
rfps = {p.stem: p.read_text(encoding="utf-8") for p in sorted((DATA / "rfp").glob("*.md"))}
print(f"{len(cvs)} CV, {len(rfps)} RFP\n")
print(cvs["01_anna_kowalska"])

14 CV, 3 RFP

# Anna Kowalska

Senior Backend Developer z 9 lat doświadczenia zawodowego, mieszka w mieście Warszawa. Absolwent(ka): Politechnika Warszawska. Dostępność od 1 stycznia 2027.

W latach 2017–2021 pracował(a) w firmie Datawise jako Backend Developer.
W latach 2021–obecnie pracował(a) w firmie Northbyte jako Senior Backend Developer.

Najmocniejsze technologie: Python (8 lat), Postgres (6 lat), Docker (5 lat). Zna także K8s oraz AWS.
Doświadczenie z pozostałymi: K8s – 3 lata, AWS – 4 lata.

Najważniejszy projekt: system płatności online (BLIK, karty) dla sieci sklepów internetowych.

Certyfikaty: AWS Certified Solutions Architect.



## 4. Ekstrakcja grafu wiedzy przez LLM

Graf można zbudować ręcznie (formularz, import z istniejącej bazy) albo automatycznie z tekstu. Przy dokumentach w języku naturalnym robi to LLM w dwóch krokach:
1. **schemat**: jakie węzły, relacje i właściwości mają znaczenie dla pytań biznesowych,
2. **ekstrakcja**: model czyta tekst i wypełnia schemat, a kod zamienia wynik na węzły i relacje.

Schemat zapisuje się klasą **Pydantic**. To klasa Pythona z typami pól, podobna do `dataclass`, ale przy tworzeniu obiektu sprawdza i konwertuje typy (zły typ = wyjątek), obsługuje modele zagnieżdżone (`list[Skill]` wewnątrz `CV`) i potrafi wygenerować JSON Schema (`CV.model_json_schema()`). Opisy pól (`Field(description=...)`) trafiają do tego schematu, więc są częścią promptu.

*Structured outputs*: klasę przekazuje się jako `response_format`, a API zwraca JSON zgodny ze schematem, od razu zamieniony na obiekt (`message.parsed`). Nie trzeba parsować tekstu ani prosić modelu o „sam JSON”.

Schemat decyduje o tym, na jakie pytania graf odpowie. Lata doświadczenia jako właściwość relacji `HAS_SKILL` pozwalają filtrować po stażu; dostępność jako data pozwala na zapytania czasowe. Opisu projektu w schemacie nie ma, więc graf nie odpowie na pytania o niego (sekcja 10).

In [7]:
from pydantic import BaseModel, Field

class Skill(BaseModel):
    name: str = Field(description="nazwa technologii tak, jak w tekście")
    years: int | None = Field(description="lata doświadczenia, jeśli podane")

class Job(BaseModel):
    company: str
    role: str
    start_year: int
    end_year: int | None = Field(description="None, jeśli praca trwa obecnie")

class CV(BaseModel):
    name: str
    city: str
    title: str
    available_from: str = Field(description="data w formacie YYYY-MM-DD")
    university: str
    skills: list[Skill]
    jobs: list[Job]
    certifications: list[str]

In [8]:
EXTRACT_PROMPT = "Wyodrębnij z CV fakty zgodnie ze schematem. Nie zgaduj: brak informacji to null."

def extract(text, schema):
    response = client.chat.completions.parse(
        model=CHAT,
        messages=[{"role": "system", "content": EXTRACT_PROMPT}, {"role": "user", "content": text}],
        response_format=schema,
    )
    return response.choices[0].message.parsed

cv = extract(cvs["01_anna_kowalska"], CV)
print(cv.model_dump_json(indent=1, exclude={"jobs", "skills"}))
print("skills:", [(s.name, s.years) for s in cv.skills])

{
 "name": "Anna Kowalska",
 "city": "Warszawa",
 "title": "Senior Backend Developer",
 "available_from": "2027-01-01",
 "university": "Politechnika Warszawska",
 "certifications": [
  "AWS Certified Solutions Architect"
 ]
}
skills: [('Python', 8), ('Postgres', 6), ('Docker', 5), ('K8s', 3), ('AWS', 4)]


Obiekt `CV` to jeszcze nie graf. Zamiana na trójki (podmiot, relacja, dopełnienie) jest jawną, deterministyczną funkcją: model tylko czyta tekst, a strukturę grafu ustala kod.

In [9]:
def to_triples(cv: CV):
    yield (cv.name, "LIVES_IN", cv.city)
    yield (cv.name, "STUDIED_AT", cv.university)
    for s in cv.skills:
        yield (cv.name, f"HAS_SKILL {{years: {s.years}}}", s.name)
    for j in cv.jobs:
        yield (cv.name, f"WORKED_AT {{from: {j.start_year}, to: {j.end_year}}}", j.company)
    for c in cv.certifications:
        yield (cv.name, "HAS_CERT", c)

for t in to_triples(cv):
    print(t)

('Anna Kowalska', 'LIVES_IN', 'Warszawa')
('Anna Kowalska', 'STUDIED_AT', 'Politechnika Warszawska')
('Anna Kowalska', 'HAS_SKILL {years: 8}', 'Python')
('Anna Kowalska', 'HAS_SKILL {years: 6}', 'Postgres')
('Anna Kowalska', 'HAS_SKILL {years: 5}', 'Docker')
('Anna Kowalska', 'HAS_SKILL {years: 3}', 'K8s')
('Anna Kowalska', 'HAS_SKILL {years: 4}', 'AWS')
('Anna Kowalska', 'WORKED_AT {from: 2017, to: 2021}', 'Datawise')
('Anna Kowalska', 'WORKED_AT {from: 2021, to: None}', 'Northbyte')
('Anna Kowalska', 'HAS_CERT', 'AWS Certified Solutions Architect')


Ekstrakcja całego zbioru. Wynik zapisuje się do `data/extracted.json`: kolejne uruchomienie notebooka nie wywołuje modelu ponownie, a bez dostępu do API można pracować na zapisanym pliku. Ponowna ekstrakcja: usunąć plik.

In [10]:
import json
from concurrent.futures import ThreadPoolExecutor

CACHE = DATA / "extracted.json"
if CACHE.exists():
    extracted = {k: CV.model_validate(v) for k, v in json.loads(CACHE.read_text(encoding="utf-8")).items()}
    print("wczytano zapisaną ekstrakcję")
else:
    with ThreadPoolExecutor(4) as pool:
        extracted = dict(zip(cvs, pool.map(lambda t: extract(t, CV), cvs.values())))
    CACHE.write_text(json.dumps({k: v.model_dump() for k, v in extracted.items()}, ensure_ascii=False, indent=1), encoding="utf-8")

print(len(extracted), "CV,", sum(len(c.skills) for c in extracted.values()), "umiejętności,",
      sum(len(c.jobs) for c in extracted.values()), "miejsc pracy")

14 CV, 63 umiejętności, 22 miejsc pracy


### Jakość ekstrakcji

Dzięki `ground_truth.json` można policzyć, ile faktów model odczytał poprawnie. W prawdziwym projekcie takiego pliku nie ma; wtedy ręcznie sprawdza się próbkę dokumentów.

In [11]:
truth = json.loads((DATA / "ground_truth.json").read_text(encoding="utf-8"))
ALIAS = {a.lower(): canon for canon, al in truth["aliases"].items() for a in al}

ok_skill = ok_years = n_skill = ok_avail = 0
for person, cv in zip(truth["people"], extracted.values()):
    got = {ALIAS.get(s.name.lower(), s.name): s.years for s in cv.skills}
    for skill, years in person["skills"].items():
        n_skill += 1
        ok_skill += skill in got
        ok_years += got.get(skill) == years
    ok_avail += cv.available_from == person["available_from"]

print(f"umiejętności: {ok_skill}/{n_skill}, lata doświadczenia: {ok_years}/{n_skill}, dostępność: {ok_avail}/{len(extracted)}")

umiejętności: 63/63, lata doświadczenia: 63/63, dostępność: 14/14


## 5. Spójność encji

Gdy dane wpisuje człowiek, formularz z listą wyboru pilnuje, żeby „Kubernetes” był zawsze zapisany tak samo. Przy ekstrakcji przez LLM model zapisuje nazwy tak, jak w tekście. Bez normalizacji „K8s” i „Kubernetes” staną się dwoma węzłami, a zapytanie „kto zna Kubernetes” pominie część osób.

Co pomaga zachować jedną encję = jeden węzeł:
- **słownik nazw kanonicznych** (`'k8s' → 'Kubernetes'`), który można przejrzeć i poprawić ręcznie,
- **lista dozwolonych wartości w schemacie** (`Literal["Python", "Java", ...]`); wtedy ekstrakcja nie znajdzie technologii spoza listy,
- **normalizacja tekstu i fuzzy matching** z istniejącymi węzłami (wielkość liter, spacje, kropki; podobieństwo napisów, np. biblioteka `rapidfuzz`),
- **istniejące nazwy w prompcie** albo wyszukiwanie najbliższej nazwy po embeddingach, żeby model wybrał węzeł, który już jest,
- **unikalny klucz i `MERGE`** przy ładowaniu (sekcja 6): ta sama nazwa nigdy nie da drugiego węzła.

Poniżej wariant ze słownikiem: LLM dostaje listę unikalnych nazw i przypisuje każdej nazwę kanoniczną.

In [12]:
from collections import Counter

raw = Counter(s.name for cv in extracted.values() for s in cv.skills)
print(len(raw), "różnych nazw umiejętności:")
print(sorted(raw))

34 różnych nazw umiejętności:
['AWS', 'Amazon Web Services', 'Apache Kafka', 'Azure', 'Docker', 'Go', 'Golang', 'JS', 'Java', 'JavaScript', 'K8s', 'Kafka', 'Kubernetes', 'ML', 'Microsoft Azure', 'Node.js', 'NodeJS', 'PostgreSQL', 'Postgres', 'PyTorch', 'Python', 'Python 3', 'React', 'React.js', 'ReactJS', 'Spring', 'Spring Boot', 'TS', 'Terraform', 'TypeScript', 'k8s', 'postgres', 'python', 'uczenie maszynowe']


In [13]:
class Alias(BaseModel):
    raw: str
    canonical: str

class AliasMap(BaseModel):
    items: list[Alias]

DEDUP_PROMPT = """Dostajesz listę nazw technologii z CV. Każdej przypisz nazwę kanoniczną:
najkrótszą powszechnie używaną nazwę technologii (np. 'k8s' -> 'Kubernetes', 'Amazon Web Services' -> 'AWS').
Różne technologie zostaw osobno. Zwróć każdą nazwę z listy dokładnie raz."""

def canonical_map(names):
    response = client.chat.completions.parse(
        model=CHAT,
        messages=[{"role": "system", "content": DEDUP_PROMPT}, {"role": "user", "content": json.dumps(sorted(names))}],
        response_format=AliasMap,
    )
    return {a.raw: a.canonical for a in response.choices[0].message.parsed.items}

CANON = canonical_map(raw)
merged = {k: v for k, v in CANON.items() if k != v}
print(f"{len(raw)} nazw -> {len(set(CANON.values()))} węzłów; zmienione:")
for k, v in sorted(merged.items(), key=lambda kv: kv[1]):
    print(f"  {k:22} -> {v}")

34 nazw -> 18 węzłów; zmienione:
  Amazon Web Services    -> AWS
  Microsoft Azure        -> Azure
  Golang                 -> Go
  JavaScript             -> JS
  Apache Kafka           -> Kafka
  K8s                    -> Kubernetes
  k8s                    -> Kubernetes
  uczenie maszynowe      -> ML
  NodeJS                 -> Node.js
  PostgreSQL             -> Postgres
  postgres               -> Postgres
  Python 3               -> Python
  python                 -> Python
  React.js               -> React
  ReactJS                -> React
  TypeScript             -> TS


In [14]:
# sprawdzenie z ground_truth: warianty jednej technologii mają trafić do jednego węzła,
# a różne technologie do różnych węzłów
groups = {}
for raw_name, canonical in CANON.items():
    groups.setdefault(ALIAS.get(raw_name.lower(), raw_name), set()).add(canonical)
split = {t: sorted(c) for t, c in groups.items() if len(c) > 1}
merged_wrong = {c for c in CANON.values() if len({t for t, cs in groups.items() if c in cs}) > 1}
print("rozbite na kilka węzłów:", split or "brak")
print("różne technologie w jednym węźle:", merged_wrong or "brak")

rozbite na kilka węzłów: {'Spring': ['Spring', 'Spring Boot']}
różne technologie w jednym węźle: brak


## 6. Ładowanie do Neo4j

Każdy węzeł ma unikalny klucz (`name`, `id`) zabezpieczony ograniczeniem `UNIQUE`. Zapytanie ładujące dostaje z Pythona parametr `$people`: listę słowników, po jednym na CV (`cv.model_dump()`). `UNWIND $people AS p` zamienia tę listę na wiersze, więc reszta zapytania wykonuje się raz dla każdej osoby, a `FOREACH` przechodzi po liście umiejętności i miejsc pracy jednej osoby. `MERGE` po kluczu łączy encje z różnych dokumentów w jeden węzeł: „Python” z ośmiu CV to jeden węzeł, a nie osiem.

RFP przechodzą tę samą drogę: schemat, ekstrakcja, normalizacja nazw.

In [15]:
class Requirement(BaseModel):
    skill: str
    min_years: int

class RFP(BaseModel):
    id: str
    title: str
    city: str
    start_date: str = Field(description="YYYY-MM-DD")
    team_size: int
    requirements: list[Requirement]

rfp_data = []
for rid, text in rfps.items():
    r = extract(text, RFP)
    r.id = rid  # klucz z nazwy pliku, nie od modelu
    rfp_data.append(r)
    for q in r.requirements:
        q.skill = CANON.get(q.skill, q.skill)
    print(r.id, r.start_date, [(q.skill, q.min_years) for q in r.requirements])

RFP-001 2027-01-01 [('Python', 4), ('Postgres', 3), ('AWS', 2)]


RFP-002 2027-03-01 [('Kubernetes', 3), ('Terraform', 2), ('Docker', 3)]


RFP-003 2027-02-01 [('React', 3), ('TS', 3), ('Node.js', 2)]


In [16]:
cypher("MATCH (n) DETACH DELETE n")
for label, key in [("Person", "name"), ("Skill", "name"), ("Company", "name"), ("City", "name"),
                   ("University", "name"), ("Certification", "name"), ("RFP", "id")]:
    cypher(f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.{key} IS UNIQUE")

people = []
for cv in extracted.values():
    p = cv.model_dump()
    for s in p["skills"]:
        s["name"] = CANON.get(s["name"], s["name"])
    people.append(p)

cypher("""
UNWIND $people AS p
MERGE (person:Person {name: p.name})
  SET person.title = p.title, person.available_from = date(p.available_from)
MERGE (city:City {name: p.city})          MERGE (person)-[:LIVES_IN]->(city)
MERGE (uni:University {name: p.university}) MERGE (person)-[:STUDIED_AT]->(uni)
FOREACH (s IN p.skills |
  MERGE (skill:Skill {name: s.name})
  MERGE (person)-[h:HAS_SKILL]->(skill) SET h.years = s.years)
FOREACH (j IN p.jobs |
  MERGE (c:Company {name: j.company})
  MERGE (person)-[w:WORKED_AT]->(c)
  SET w.role = j.role, w.from = j.start_year, w.to = coalesce(j.end_year, 2026))
FOREACH (cert IN p.certifications |
  MERGE (c:Certification {name: cert}) MERGE (person)-[:HAS_CERT]->(c))
""", people=people)

cypher("""
UNWIND $rfps AS r
MERGE (rfp:RFP {id: r.id})
  SET rfp.title = r.title, rfp.start_date = date(r.start_date), rfp.team_size = r.team_size
MERGE (city:City {name: r.city}) MERGE (rfp)-[:LOCATED_IN]->(city)
FOREACH (q IN r.requirements |
  MERGE (s:Skill {name: q.skill})
  MERGE (rfp)-[req:REQUIRES]->(s) SET req.min_years = q.min_years)
""", rfps=[r.model_dump() for r in rfp_data])

for row in cypher("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC"):
    print(f"{row['label']:14} {row['n']}")
print("relacje:", cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

Skill          18
Person         14
University     9
Company        6
City           5
Certification  4
RFP            3


relacje: 131


## 7. Zapytania do grafu TalentMatch

Zapytania, które w grafie są jednym `MATCH`, a w zbiorze fragmentów tekstu wymagałyby przeczytania wszystkich dokumentów naraz.

In [17]:
# zliczanie i agregacja
print(cypher("""
MATCH (p:Person)-[r:HAS_SKILL]->(s:Skill)
RETURN s.name AS skill, count(p) AS people, round(avg(r.years), 1) AS avg_years
ORDER BY people DESC LIMIT 5
"""))

# relacja wielokrokowa: kto pracował w tej samej firmie co Anna Kowalska
print(cypher("""
MATCH (:Person {name: 'Anna Kowalska'})-[:WORKED_AT]->(c:Company)<-[:WORKED_AT]-(p:Person)
RETURN c.name AS company, collect(p.name) AS colleagues
"""))

[{'skill': 'Python', 'people': 8, 'avg_years': 4.8}, {'skill': 'Kubernetes', 'people': 8, 'avg_years': 3.6}, {'skill': 'Docker', 'people': 7, 'avg_years': 4.3}, {'skill': 'Postgres', 'people': 6, 'avg_years': 3.8}, {'skill': 'AWS', 'people': 5, 'avg_years': 4.0}]
[{'company': 'Northbyte', 'colleagues': ['Tomasz Wójcik', 'Bartosz Krawczyk', 'Joanna Kozłowska']}, {'company': 'Datawise', 'colleagues': ['Paweł Dąbrowski', 'Natalia Mazur', 'Katarzyna Wiśniewska']}]


### Trudniejsze zapytanie: dopasowanie do RFP

Zapytanie łączy wymagania RFP, umiejętności z latami doświadczenia i dostępność. Kandydat może mieć co najwyżej jedno niespełnione wymaganie; wynik zawiera powód, dla którego ktoś odpada. Kolejne kroki:

1. `MATCH (rfp:RFP {id: $rfp})-[req:REQUIRES]->(s:Skill)`: wymagania RFP o identyfikatorze z parametru `$rfp`,
2. `WITH rfp, collect({skill: …, min: …}) AS reqs`: wszystkie wymagania zebrane w jedną listę słowników (jeden wiersz zamiast trzech),
3. `MATCH (p:Person)`: każda osoba z każdą listą wymagań,
4. `[r IN reqs | r {.*, have: …}]`: dla każdego wymagania dopisuje do słownika staż osoby; `[(p)-[h:HAS_SKILL]->(:Skill {name: r.skill}) | h.years][0]` to lata z relacji albo `null`, a `coalesce(…, 0)` zamienia `null` na 0,
5. `size([c IN checks WHERE c.have >= c.min])`: liczba spełnionych wymagań; `available` porównuje daty,
6. `WHERE met >= size(checks) - 1`, `RETURN`, `ORDER BY`: filtr, kolumny wyniku i kolejność; `missing` to lista braków.

Taka jawna logika (reguły, progi, wagi w kodzie) jest sprawdzalna: wiadomo, dlaczego ktoś jest wyżej. LLM nie podejmuje tu decyzji.

In [18]:
MATCH_QUERY = """
MATCH (rfp:RFP {id: $rfp})-[req:REQUIRES]->(s:Skill)
WITH rfp, collect({skill: s.name, min: req.min_years}) AS reqs
MATCH (p:Person)
WITH rfp, reqs, p, [r IN reqs |
       r {.*, have: coalesce([(p)-[h:HAS_SKILL]->(:Skill {name: r.skill}) | h.years][0], 0)}] AS checks
WITH rfp, p, checks,
     size([c IN checks WHERE c.have >= c.min]) AS met,
     p.available_from <= rfp.start_date AS available
WHERE met >= size(checks) - 1
RETURN p.name AS name, met, size(checks) AS required, available,
       [c IN checks WHERE c.have < c.min | c.skill + ' ' + c.have + '/' + c.min] AS missing
ORDER BY met DESC, available DESC
"""
for row in cypher(MATCH_QUERY, rfp="RFP-002"):
    print(row)

{'name': 'Piotr Nowak', 'met': 3, 'required': 3, 'available': True, 'missing': []}
{'name': 'Agnieszka Zielińska', 'met': 3, 'required': 3, 'available': False, 'missing': []}
{'name': 'Anna Kowalska', 'met': 2, 'required': 3, 'available': True, 'missing': ['Terraform 0/2']}
{'name': 'Ewa Woźniak', 'met': 2, 'required': 3, 'available': True, 'missing': ['Terraform 0/2']}
{'name': 'Bartosz Krawczyk', 'met': 2, 'required': 3, 'available': False, 'missing': ['Docker 0/3']}


## 8. Agentic GraphRAG

Model nie musi odpowiadać z pamięci ani z kilku pobranych fragmentów tekstu. Dostaje narzędzie `run_cypher`, a w system prompcie **schemat grafu**: jakie są węzły, relacje i właściwości. Sam pisze zapytania Cypher, wykonuje je przez narzędzie, czyta wyniki i decyduje, czy potrzebuje kolejnego zapytania. To ta sama pętla ReAct (myśl → akcja → obserwacja) co na zajęciach 3, tylko narzędziem jest baza grafowa.

Bezpieczeństwo: zapytanie od modelu wykonuje się w transakcji **tylko do odczytu** (`run_read`), więc wygenerowane `DELETE` kończy się błędem. Błąd składni też wraca do modelu jako tekst: model może poprawić zapytanie w następnym kroku.

In [19]:
SCHEMA = """
(:Person {name, title, available_from: date})
(:Person)-[:HAS_SKILL {years: int}]->(:Skill {name})
(:Person)-[:WORKED_AT {role, from: int, to: int}]->(:Company {name})   // to = 2026: praca trwa
(:Person)-[:LIVES_IN]->(:City {name})
(:Person)-[:STUDIED_AT]->(:University {name})
(:Person)-[:HAS_CERT]->(:Certification {name})
(:RFP {id, title, start_date: date, team_size})-[:REQUIRES {min_years: int}]->(:Skill)
(:RFP)-[:LOCATED_IN]->(:City)
Nazwy umiejętności: """ + ", ".join(sorted(set(CANON.values())))

AGENT_SYSTEM = f"""Odpowiadasz na pytania o kandydatów i zapytania ofertowe (RFP) na podstawie grafu w Neo4j.
Dane pobierasz wyłącznie narzędziem run_cypher (Cypher, Neo4j 5, tylko odczyt). Schemat grafu:
{SCHEMA}
Właściwości relacji odczytujesz przez zmienną relacji, np. h.years w (p)-[h:HAS_SKILL]->(s).
Przed każdym wywołaniem narzędzia piszesz jedno krótkie zdanie: co sprawdzasz.
Gdy zapytanie zwróci błąd, poprawiasz je. Gdy w grafie nie ma potrzebnych danych, mówisz to wprost.
Odpowiadasz krótko, po polsku, zwykłym tekstem."""

In [20]:
from langchain.tools import tool
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI, AzureChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def run_cypher(query: str) -> str:
    """Wykonuje zapytanie Cypher (tylko odczyt) w grafie TalentMatch. Zwraca wiersze wyniku jako JSON albo komunikat błędu."""
    try:
        rows = run_read(query)
    except Exception as e:
        return f"błąd: {str(e).splitlines()[0][:300]}"
    return json.dumps(rows[:30], ensure_ascii=False, default=str)

def make_llm(model):
    if PROVIDER == "azure":
        return AzureChatOpenAI(azure_deployment=model, api_version="2024-12-01-preview")
    return ChatOpenAI(model=model, base_url=str(client.base_url), api_key=client.api_key)

llm_with_tools = make_llm(AGENT).bind_tools([run_cypher], parallel_tool_calls=False)

def call_model(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([SystemMessage(AGENT_SYSTEM)] + state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode([run_cypher]))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)   # tool_calls? -> "tools" : END
builder.add_edge("tools", "model")
agent = builder.compile()

Funkcja `ask` uruchamia agenta i wypisuje każdy krok pętli: ile wiadomości poszło do modelu, myśl, wywołanie narzędzia (zapytanie Cypher) i obserwację (wynik z bazy).

In [21]:
def ask(question, verbose=True):
    messages = agent.invoke({"messages": [("user", question)]}, {"recursion_limit": 20})["messages"]
    step = 0
    for i, m in enumerate(messages):
        if m.type == "ai":
            step += 1
            if verbose:
                print(f"[{step}] wysłano {i + 1} wiadomości, {m.usage_metadata['input_tokens']} tokenów")
                if m.tool_calls and m.content:
                    print(f"    myśl: {m.content.strip()}")
                for c in m.tool_calls:
                    print(f"    akcja: run_cypher({' '.join(c['args']['query'].split())})")
                if not m.tool_calls:
                    print(f"    odpowiedź: {m.content.strip()}")
        elif m.type == "tool" and verbose:
            print(f"    obserwacja: {m.content[:160]}{'…' if len(m.content) > 160 else ''}")
    return messages

messages = ask("Kto zna Kubernetes co najmniej 3 lata i jest dostępny przed startem RFP-002?")

[1] wysłano 2 wiadomości, 569 tokenów
    myśl: Sprawdzam kandydatów z co najmniej 3 lata doświadczenia w Kubernetes i dostępnych przed startem RFP-002.
    akcja: run_cypher(MATCH (r:RFP {id:'RFP-002'}) MATCH (p:Person)-[h:HAS_SKILL]->(:Skill {name:'Kubernetes'}) WHERE h.years >= 3 AND p.available_from < r.start_date RETURN p.name AS name, p.title AS title, p.available_from AS available_from, h.years AS years_kubernetes, r.start_date AS rfp_start_date ORDER BY h.years DESC, p.available_from ASC;)
    obserwacja: [{"name": "Piotr Nowak", "title": "DevOps Engineer", "available_from": "2026-11-01", "years_kubernetes": 5, "rfp_start_date": "2027-03-01"}, {"name": "Anna Kowa…
[2] wysłano 4 wiadomości, 894 tokenów
    odpowiedź: Kandydaci spełniający kryteria (Kubernetes ≥ 3 lata oraz dostępni przed startem **RFP-002**):

- **Piotr Nowak** (DevOps Engineer) — Kubernetes: **5 lat**, dostępny od **2026-11-01** (RFP start: **2027-03-01**)
- **Anna Kowalska** (Senior Backend Developer) — Kubern

In [22]:
# zapytanie zapisujące w transakcji tylko do odczytu
print(run_cypher.invoke({"query": "MATCH (n) DETACH DELETE n"}))

błąd: {neo4j_code: Neo.ClientError.Statement.AccessMode} {message: Writing in read access mode not allowed. Attempted write to neo4j} {gql_status: 08N03} {gql_status_description: error: connection exception - failed to write to graph. Failed to write to graph `neo4j`. Check the defined access mode in both


## 9. RAG i structured RAG na tych samych CV

Punkt odniesienia: RAG z zajęć 2. Każde CV to jeden fragment (są krótkie), embeddingi w bazie Chroma, wyszukanie `k=4` najbliższych CV i odpowiedź modelu na ich podstawie.

Przy każdym CV zapisuje się też **metadane** z ekstrakcji: miasto, stanowisko, dostępność. **Structured RAG** najpierw filtruje po metadanych, a dopiero potem szuka wektorowo wśród tego, co zostało. Filtr układa LLM ze structured outputs (tzw. *self-querying*): z pytania „Kto z Krakowa…” wyciąga `city = 'Kraków'`, a resztę pytania zostawia do wyszukiwania semantycznego. Tak samo działa filtrowanie w Chroma (`where`), pgvector (`WHERE` w SQL obok operatora odległości), Azure AI Search czy Azure Cosmos DB.

In [23]:
import chromadb

def embed(texts):
    return [d.embedding for d in embed_client.embeddings.create(model=EMBED, input=texts).data]

db = chromadb.Client()
if "talentmatch" in [c.name for c in db.list_collections()]:
    db.delete_collection("talentmatch")
col = db.create_collection("talentmatch", metadata={"hnsw:space": "cosine"})
col.add(
    ids=list(cvs),
    documents=list(cvs.values()),
    embeddings=embed(list(cvs.values())),
    metadatas=[{"name": e.name, "city": e.city, "title": e.title,
                "available_from": int(e.available_from.replace("-", ""))} for e in extracted.values()],
)

ANSWER_PROMPT = "Odpowiedz krótko po polsku wyłącznie na podstawie dokumentów. Liczby i listy osób podawaj dokładnie."

def answer_from(question, docs):
    context = "\n\n---\n\n".join(docs)
    return client.chat.completions.create(
        model=AGENT,
        messages=[{"role": "system", "content": ANSWER_PROMPT},
                  {"role": "user", "content": f"Pytanie: {question}\nDokumenty:\n{context}"}],
    ).choices[0].message.content

def naive_rag(question, k=4):
    res = col.query(query_embeddings=embed([question]), n_results=k)
    hits = [(m["name"], round(1 - d, 3)) for m, d in zip(res["metadatas"][0], res["distances"][0])]
    return hits, answer_from(question, res["documents"][0])

hits, answer = naive_rag("Ile osób zna Kubernetes?")
print(hits, "\n")
print(answer)

[('Piotr Nowak', 0.477), ('Bartosz Krawczyk', 0.465), ('Agnieszka Zielińska', 0.434), ('Joanna Kozłowska', 0.411)] 

Z dokumentów wynika, że **4 osoby** znają Kubernetes: **Piotr Nowak, Bartosz Krawczyk, Agnieszka Zielińska, Joanna Kozłowska**.


In [24]:
class Filter(BaseModel):
    city: str | None = Field(description="miasto, jeśli pytanie dotyczy osób z konkretnego miasta")
    available_before: str | None = Field(description="YYYY-MM-DD, jeśli pytanie dotyczy dostępności przed datą")
    query: str = Field(description="treść do wyszukania semantycznego, bez warunków przeniesionych do filtra")

FILTER_PROMPT = "Rozłóż pytanie o kandydatów na filtr metadanych i zapytanie do wyszukiwarki semantycznej. Nie dodawaj warunków, których nie ma w pytaniu."

def make_where(f: Filter):
    conds = []
    if f.city:
        conds.append({"city": f.city})
    if f.available_before:
        conds.append({"available_from": {"$lt": int(f.available_before.replace("-", ""))}})
    return None if not conds else conds[0] if len(conds) == 1 else {"$and": conds}

def structured_rag(question, k=4):
    f = client.chat.completions.parse(
        model=CHAT, response_format=Filter,
        messages=[{"role": "system", "content": FILTER_PROMPT}, {"role": "user", "content": question}],
    ).choices[0].message.parsed
    where = make_where(f)
    res = col.query(query_embeddings=embed([f.query]), n_results=k, where=where)
    hits = [(m["name"], round(1 - d, 3)) for m, d in zip(res["metadatas"][0], res["distances"][0])]
    return where, f.query, hits, answer_from(question, res["documents"][0])

where, query, hits, answer = structured_rag("Kto z Krakowa pracował przy systemach płatności?")
print("filtr:", where, "| zapytanie:", query)
print(hits, "\n")
print(answer)

filtr: {'city': 'Kraków'} | zapytanie: kandydaci z Krakowa, którzy pracowali przy systemach płatności
[('Marcin Jankowski', 0.517), ('Piotr Nowak', 0.499), ('Michał Lewandowski', 0.431)] 

Z Krakowa przy systemach płatności pracował **Marcin Jankowski** (przetwarzanie strumienia transakcji płatniczych pod kątem wykrywania nadużyć) oraz **Piotr Nowak** (infrastruktura bramki płatniczej operatora kart).


## 10. Porównanie: RAG, structured RAG i GraphRAG

Dziewięć pytań różnych typów. Poprawne odpowiedzi wylicza kod z `ground_truth.json`. Zgodność sprawdza prosta funkcja: czy odpowiedź zawiera poprawną liczbę albo dokładnie te osoby, które powinna. Przy odpowiedziach w formie listy i liczby jest to pewniejsze niż *LLM as a judge* (zajęcia 2).

In [25]:
P = truth["people"]
def has(p, skill, years=1):
    return p["skills"].get(skill, 0) >= years

python_years = [p["skills"]["Python"] for p in P if has(p, "Python")]
anna_companies = {j["company"] for j in P[0]["jobs"]}
rfp2 = next(r for r in truth["rfps"] if r["id"] == "RFP-002")

QUESTIONS = [
    ("lookup", "Jakie certyfikaty ma Piotr Nowak?",
     next(p["certifications"] for p in P if p["name"] == "Piotr Nowak")),
    ("opis projektu", "Kto pracował nad wykrywaniem nadużyć w transakcjach płatniczych?",
     [p["name"] for p in P if "nadużyć" in p["project"]["text"]]),
    ("zliczanie", "Ile osób zna Kubernetes?",
     sum(has(p, "Kubernetes") for p in P)),
    ("filtrowanie", "Kto z Warszawy zna Pythona?",
     [p["name"] for p in P if p["city"] == "Warszawa" and has(p, "Python")]),
    ("agregacja", "Ile średnio lat doświadczenia z Pythonem mają osoby, które go znają?",
     round(sum(python_years) / len(python_years), 2)),
    ("wielokrokowe", "Kto pracował w tej samej firmie co Anna Kowalska?",
     [p["name"] for p in P[1:] if anna_companies & {j["company"] for j in p["jobs"]}]),
    ("dopasowanie", "Kto spełnia wszystkie wymagania RFP-002 i jest dostępny przed startem projektu?",
     [p["name"] for p in P if all(has(p, s, y) for s, y in rfp2["requirements"].items())
      and p["available_from"] <= rfp2["start"]]),
    ("filtr + treść", "Kto z Krakowa pracował przy systemach płatności?",
     [p["name"] for p in P if p["city"] == "Kraków" and p["project"]["domain"] == "płatności"]),
    ("filtr + treść", "Kto z Gdańska pracował przy projektach medycznych?",
     [p["name"] for p in P if p["city"] == "Gdańsk" and p["project"]["domain"] == "medycyna"]),
]
for kind, q, t in QUESTIONS:
    print(f"{kind:14} {q}\n{'':14} -> {t}")

lookup         Jakie certyfikaty ma Piotr Nowak?
               -> ['Certified Kubernetes Administrator']
opis projektu  Kto pracował nad wykrywaniem nadużyć w transakcjach płatniczych?
               -> ['Marcin Jankowski']
zliczanie      Ile osób zna Kubernetes?
               -> 8
filtrowanie    Kto z Warszawy zna Pythona?
               -> ['Anna Kowalska', 'Natalia Mazur']
agregacja      Ile średnio lat doświadczenia z Pythonem mają osoby, które go znają?
               -> 4.75
wielokrokowe   Kto pracował w tej samej firmie co Anna Kowalska?
               -> ['Katarzyna Wiśniewska', 'Tomasz Wójcik', 'Paweł Dąbrowski', 'Joanna Kozłowska', 'Natalia Mazur', 'Bartosz Krawczyk']
dopasowanie    Kto spełnia wszystkie wymagania RFP-002 i jest dostępny przed startem projektu?
               -> ['Piotr Nowak']
filtr + treść  Kto z Krakowa pracował przy systemach płatności?
               -> ['Piotr Nowak', 'Marcin Jankowski']
filtr + treść  Kto z Gdańska pracował przy projektach medycznych

In [26]:
import re

NAMES = [p["name"] for p in P]
WORDS = {1: "jed", 2: "dw", 3: "trz", 4: "czter", 5: "pięć", 6: "sześć", 7: "siedem", 8: "osiem"}
stem = lambda name: name.split()[1][:5]  # „Kowalska” i „Kowalskiej” mają wspólny początek

def judge(question, expected, answer):
    """Odpowiedź poprawna, gdy zawiera oczekiwaną liczbę albo dokładnie oczekiwane osoby."""
    if isinstance(expected, (int, float)):
        num = str(expected).replace(".", "[.,]")
        return bool(re.search(rf"(?<![\d,.]){num}(?![\d,]|\.\d)", answer)) or WORDS.get(expected, "#") in answer.lower()
    if expected and set(expected) <= set(NAMES):
        found = {n for n in NAMES if stem(n) in answer and stem(n) not in question}
        return found == set(expected)
    return all(e.lower() in answer.lower() for e in expected)

def graph_rag(question):
    messages = ask(question, verbose=False)
    calls = [c["args"]["query"] for m in messages if m.type == "ai" for c in m.tool_calls]
    results = [m.content for m in messages if m.type == "tool"]
    return calls, results, messages[-1].content

results = []
for kind, q, expected in QUESTIONS:
    hits, rag_answer = naive_rag(q)
    where, query, s_hits, s_answer = structured_rag(q)
    calls, observations, g_answer = graph_rag(q)
    results.append({"kind": kind, "question": q, "expected": expected,
                    "rag": {"hits": hits, "answer": rag_answer, "correct": judge(q, expected, rag_answer)},
                    "structured": {"where": where, "query": query, "hits": s_hits, "answer": s_answer,
                                   "correct": judge(q, expected, s_answer)},
                    "graph": {"cypher": calls, "observations": observations, "answer": g_answer,
                              "correct": judge(q, expected, g_answer)}})

mark = {True: "✓", False: "✗"}
print(f"{'typ':14} {'RAG':>4} {'Struct':>7} {'Graph':>6}")
for r in results:
    print(f"{r['kind']:14} {mark[r['rag']['correct']]:>4} {mark[r['structured']['correct']]:>7} {mark[r['graph']['correct']]:>6}")
print(f"{'razem':14} {sum(r['rag']['correct'] for r in results):>4} "
      f"{sum(r['structured']['correct'] for r in results):>7} {sum(r['graph']['correct'] for r in results):>6}")

typ             RAG  Struct  Graph
lookup            ✓       ✓      ✓
opis projektu     ✓       ✓      ✗
zliczanie         ✗       ✗      ✓
filtrowanie       ✗       ✓      ✓
agregacja         ✗       ✗      ✓
wielokrokowe      ✗       ✗      ✓
dopasowanie       ✗       ✗      ✓
filtr + treść     ✗       ✓      ✗
filtr + treść     ✓       ✓      ✗
razem             3       5      6


In [27]:
# zapis wyników dla strony z prezentacją
(DATA / "results.json").write_text(json.dumps(
    {"provider": PROVIDER, "chat": CHAT, "agent": AGENT, "embed": EMBED, "results": results,
     "canonical": CANON, "rfps": [r.model_dump() for r in rfp_data]},
    ensure_ascii=False, indent=1, default=str), encoding="utf-8")

19725